# Akan-BPE — 2A4 Llama BPB diagnostic & truncation-corrected eval

Helper notebook for **Phase 2A4** (`meta-llama/Llama-3.2-1B`). The 2A4 run was the only
ladder rung where the Akan fine-tune appeared to *lose* to the base on bits-per-byte
(mean-subword **0.8966** vs an unusually low base **0.7685**, report §11).

This notebook tests the hypothesis that the low Llama base BPB is a **truncation artifact**
of the metric, not real Twi-modeling skill:

- `_build_causal_example` truncates each text to `max_length-1` tokens, but
  `compute_model_bpb` divides the (truncated) NLL by the **full** UTF-8 byte count of the text.
- The eval texts are multi-sentence paragraphs, so longer-than-budget texts have their BPB
  **underestimated — worse for higher-fertility tokenizers** (they hit the token budget after
  fewer words). Llama's base tokenizer is the least efficient on Twi (~3.07 tok/word), so its
  base BPB is deflated the most.

**Plan:** (A) setup → (B) load eval + tokenizers → (C) quantify truncation →
(D) reproduce the 0.7685 artifact → (E) corrected **base** BPB (full byte coverage, no
adapters) → (F) retrain both Llama arms in-session → (G) corrected **experiment** BPB +
corrected improvement → (H) summary / verdict.

Runs top-to-bottom on a Kaggle single T4. `meta-llama/Llama-3.2-1B` and `google/gemma-3-1b-pt`
are gated — accept their licenses and provide an HF token (cell 3).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/2a4_llama_eval_helper.ipynb)  [![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/2a4_llama_eval_helper.ipynb)

## A. Setup

In [8]:
# 1. Clone repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

# Guard against triple-nesting on cell re-run: only clone+cd when we are not
# already sitting inside the repo root.
if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

Working directory: /kaggle/working/akan-bpe


In [9]:
# 2. Install dependencies
%pip install -q -e ".[dev,train]"
%pip install -q bitsandbytes sentencepiece pandas

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for akan-bpe (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Hugging Face authentication — Llama and Gemma are gated models.
# Accept the licenses at https://huggingface.co/meta-llama/Llama-3.2-1B and
# https://huggingface.co/google/gemma-3-1b-pt first, then run this cell and paste
# a token with access (or set the HF_TOKEN env var / Kaggle Secret).
from huggingface_hub import login
login()

In [10]:
# 3. Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime / Settings -> Accelerator -> GPU T4, then re-run."
    )
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [11]:
# 4. Download Akan datasets from HuggingFace Hub
# --tts-limit 50000 pins the pristine-Twi corpus to the same 45,000/2,500/2,500
# split used in the report (the default would pull ~188k rows and shift the split).
!python scripts/download.py --output-dir data --tts-limit 50000

Resolving data files: 100%|████████████████| 270/270 [00:00<00:00, 33512.73it/s]
Wrote 8085 rows to data/aka_asr_train.jsonl
Wrote 1011 rows to data/aka_asr_validation.jsonl
Wrote 1011 rows to data/aka_asr_test.jsonl
README.md: 4.06kB [00:00, 2.14MB/s]
Wrote 40000 rows to data/pristine_twi_train.jsonl
Wrote 5000 rows to data/pristine_twi_validation.jsonl
Wrote 5000 rows to data/pristine_twi_test.jsonl
Manifest written to data/download_manifest.json


In [12]:
# 5. Train TTS tokenizer (if not already present)
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    print("TTS tokenizer not found - training now ...")
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    sz = Path("models/tts_tokenizer.json").stat().st_size / 1e6
    print(f"TTS tokenizer already present ({sz:.1f} MB), skipping.")

TTS tokenizer already present (0.5 MB), skipping.


In [13]:
# 6. Verify all required inputs exist
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")
for name, p in required.items():
    print(f"  {name}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")

All inputs ready.
  TTS train data: data/pristine_twi_train.jsonl  (64.0 MB)
  TTS test data: data/pristine_twi_test.jsonl  (8.0 MB)
  TTS tokenizer: models/tts_tokenizer.json  (0.5 MB)


## B. Load eval texts + tokenizers

The full **2,500-row** pristine-Twi test set (the 2A4 run only scored a 512-row cap).
`load_texts` returns the untruncated texts; `load_experiment_tokenizer` loads the Akan TTS
tokenizer exactly as the runs do.

In [14]:
import math
import numpy as np
import pandas as pd

from akan_bpe.model_integration import (
    load_texts,
    load_experiment_tokenizer,
    compute_model_bpb,        # original (truncating) BPB - used to reproduce the artifact
    build_text_dataset,       # original fixed-width dataset builder (truncates)
    _load_base_model_for_bpb, # quantized base-model loader
    _import_training_stack,   # torch + AutoModelForCausalLM + BitsAndBytesConfig + PeftModel
    _set_model_token_config,
)
from akan_bpe.metrics import count_utf8_bytes, bits_per_byte
from transformers import AutoTokenizer

# Shared runtime stack (use this torch everywhere so device handling matches the library).
stack = _import_training_stack()
torch = stack["torch"]
AutoModelForCausalLM = stack["AutoModelForCausalLM"]
BitsAndBytesConfig = stack["BitsAndBytesConfig"]
PeftModel = stack["PeftModel"]

MAX_LENGTH = 256   # the token budget used by the original runs (truncating path)
CHUNK_SIZE = 256   # full-coverage scoring chunk size (matches training context)

EVAL_FILE = Path("data/pristine_twi_test.jsonl")
AKAN_TOK_PATH = Path("models/tts_tokenizer.json")

eval_texts = load_texts(EVAL_FILE)                 # full 2,500 rows, untruncated
EVAL_BYTES = count_utf8_bytes(eval_texts)
print(f"Eval texts        : {len(eval_texts):,}")
print(f"Total UTF-8 bytes : {EVAL_BYTES:,}")

akan_tok = load_experiment_tokenizer(AKAN_TOK_PATH)

LLAMA_ID = "meta-llama/Llama-3.2-1B"
BASE_MODELS = {
    "Llama-3.2-1B": LLAMA_ID,
    "Qwen3-1.7B": "Qwen/Qwen3-1.7B",
    "gemma-3-1b-pt": "google/gemma-3-1b-pt",
}

# Base tokenizers for the truncation diagnosis. Tolerant: a gated/unavailable model is
# skipped rather than killing the notebook (Llama is the focus).
base_toks = {}
for name, mid in BASE_MODELS.items():
    try:
        base_toks[name] = AutoTokenizer.from_pretrained(mid)
    except Exception as exc:  # noqa: BLE001
        print(f"  [skip] base tokenizer {name} ({mid}): {exc}")
print("Loaded base tokenizers:", list(base_toks))

Eval texts        : 5,000
Total UTF-8 bytes : 7,597,984


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Loaded base tokenizers: ['Llama-3.2-1B', 'Qwen3-1.7B', 'gemma-3-1b-pt']


## C. Truncation diagnosis (no model load)

For each tokenizer over the **full** eval set: token-length distribution, the **fraction of
texts truncated** at `max_length=256`, and the **covered-byte fraction** — the share of each
text's bytes that the original (truncating) BPB actually scored. A low covered-byte fraction
means the original BPB divided real NLL by bytes it never scored, deflating it.

In [15]:
def truncation_stats(tokenizer, texts, max_length):
    """Token-length distribution + truncation/coverage stats for one tokenizer."""
    budget = max_length - 1  # _build_causal_example keeps max_length-1 tokens, then EOS
    tok_lens = []
    covered_bytes = 0
    full_bytes = 0
    for t in texts:
        ids = tokenizer(t, add_special_tokens=False)["input_ids"]
        tok_lens.append(len(ids))
        fb = len(t.encode("utf-8"))
        full_bytes += fb
        if len(ids) > budget:
            cov_text = tokenizer.decode(ids[:budget])
            covered_bytes += min(len(cov_text.encode("utf-8")), fb)
        else:
            covered_bytes += fb
    a = np.array(tok_lens)
    return {
        "mean_tokens": round(float(a.mean()), 1),
        "median_tokens": int(np.median(a)),
        "p90_tokens": int(np.percentile(a, 90)),
        "max_tokens": int(a.max()),
        "pct_truncated": round(100.0 * float((a > budget).mean()), 1),
        "covered_byte_frac": round(covered_bytes / full_bytes, 4) if full_bytes else 0.0,
    }


rows = {"Akan-TTS": truncation_stats(akan_tok, eval_texts, MAX_LENGTH)}
for name, tok in base_toks.items():
    rows[name] = truncation_stats(tok, eval_texts, MAX_LENGTH)

diag = pd.DataFrame(rows).T
print("Truncation diagnosis at max_length =", MAX_LENGTH)
diag

Truncation diagnosis at max_length = 256


,mean_tokens,median_tokens,p90_tokens,max_tokens,pct_truncated,covered_byte_frac
Akan-TTS,352.6,346.0,436.0,1861.0,93.5,0.7439
Llama-3.2-1B,856.9,838.0,1068.0,5087.0,99.8,0.3070
Qwen3-1.7B,705.8,690.0,875.0,4089.0,99.5,0.3668
gemma-3-1b-pt,637.4,623.0,789.0,3739.0,99.4,0.4083


### Full-coverage BPB helper

`compute_model_bpb_full` scores **100% of every text's bytes** via consecutive
non-overlapping token chunks (no truncation). It reuses the same causal-shift sum-NLL math as
`akan_bpe.model_integration.compute_model_bpb` and the same `count_utf8_bytes` / `bits_per_byte`
primitives, so it is consistent with the library metric — only the coverage differs. Because
every model scores all bytes of the same texts, base and experiment BPB stay comparable.

Non-overlapping chunks lose left-context at each chunk boundary — a small effect that is
symmetric across models; `chunk_size` is a parameter and we cross-check 256 vs 512 below.

In [16]:
def compute_model_bpb_full(model, texts, tokenizer, torch, chunk_size=256):
    """Full byte-coverage BPB via non-overlapping chunks.

    Every text's bytes appear in the denominator. ~1/chunk_size boundary tokens
    per chunk have no left-context (cold start); this effect is symmetric across
    models. Single-token final chunks (len(ids) % chunk_size == 1) are skipped
    — affects ~0.4% of texts symmetrically across models.
    Use compute_model_bpb_sliding for a strided alternative without cold-start bias.
    """
    cross_entropy = torch.nn.CrossEntropyLoss(reduction="sum")
    total_nll_nats = 0.0
    num_target_tokens = 0
    eos_id = tokenizer.eos_token_id
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            if eos_id is not None:
                ids = ids + [eos_id]
            for start in range(0, len(ids), chunk_size):
                chunk = ids[start:start + chunk_size]
                if len(chunk) < 2:
                    continue  # single orphan token: ~0.4% of texts, symmetric
                input_ids = torch.tensor([chunk], device=model.device)
                outputs = model(input_ids=input_ids)
                shift_logits = outputs.logits[:, :-1, :]
                shift_labels = input_ids[:, 1:]
                loss = cross_entropy(
                    shift_logits.reshape(-1, shift_logits.size(-1)).float(),
                    shift_labels.reshape(-1),
                )
                total_nll_nats += float(loss.item())
                num_target_tokens += int(shift_labels.numel())
    total_bytes = count_utf8_bytes(texts)
    return {
        "bits_per_byte": bits_per_byte(total_nll_nats, total_bytes),
        "total_nll_bits": total_nll_nats / math.log(2),
        "total_bytes": total_bytes,
        "num_target_tokens": num_target_tokens,
    }


def compute_model_bpb_sliding(model, texts, tokenizer, torch, window=512, stride=256):
    """Strided sliding-window BPB: each target scored exactly once.

    Every new target has at least (window - stride - 1) tokens of left context
    within its window, eliminating the chunk-boundary cold-start of
    compute_model_bpb_full. window=512, stride=256 gives >=255 tokens of context
    per new target. 100% of bytes appear in the denominator; each target position
    is counted exactly once (no double-counting).
    """
    cross_entropy = torch.nn.CrossEntropyLoss(reduction="sum", ignore_index=-100)
    total_nll_nats = 0.0
    num_target_tokens = 0
    eos_id = tokenizer.eos_token_id
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            if eos_id is not None:
                ids = ids + [eos_id]
            n = len(ids)
            if n < 2:
                continue
            # scored_end: exclusive upper bound of target positions already counted.
            # Target positions in ids are [1, n); scored_end starts at 1 (nothing counted).
            scored_end = 1
            for begin in range(0, n - 1, stride):
                end = min(begin + window, n)
                chunk = ids[begin:end]
                if len(chunk) < 2:
                    break
                inp = torch.tensor([chunk], device=model.device)
                outputs = model(input_ids=inp)
                shift_logits = outputs.logits[:, :-1, :]
                # shift_labels[0, j] is the target for ids[begin + j + 1].
                shift_labels = inp[:, 1:].clone()
                # Mask targets already counted in a previous window.
                n_mask = max(0, scored_end - begin - 1)
                if n_mask > 0:
                    shift_labels[:, :n_mask] = -100
                valid = int((shift_labels != -100).sum().item())
                if valid > 0:
                    loss = cross_entropy(
                        shift_logits.reshape(-1, shift_logits.size(-1)).float(),
                        shift_labels.reshape(-1),
                    )
                    total_nll_nats += float(loss.item())
                    num_target_tokens += valid
                scored_end = end
                if end == n:
                    break
    total_bytes = count_utf8_bytes(texts)
    return {
        "bits_per_byte": bits_per_byte(total_nll_nats, total_bytes),
        "total_nll_bits": total_nll_nats / math.log(2),
        "total_bytes": total_bytes,
        "num_target_tokens": num_target_tokens,
    }


def free():
    """Collect garbage and clear the CUDA cache. Caller must `del` its own model
    reference *before* calling this (Python can only release the object once no name
    binds it)."""
    import gc
    gc.collect()
    torch.cuda.empty_cache()

## D. Reproduce the 0.7685 artifact

Score base Llama with the **original truncating** `compute_model_bpb` on the same capped config
as the run (first **512** texts, `max_length=256`). This should reproduce the reported base BPB
of ~**0.7685**, confirming the notebook faithfully replicates the pipeline before we correct it.

In [17]:
llama_base_tok = base_toks.get("Llama-3.2-1B") or AutoTokenizer.from_pretrained(LLAMA_ID)
llama_base = _load_base_model_for_bpb(LLAMA_ID, AutoModelForCausalLM, torch, quantized=True)
_set_model_token_config(llama_base, llama_base_tok)  # align pad/eos/bos config (mirrors compute_bpb_metrics)

# Original method, run config: 512 texts, max_length 256 (truncating), full bytes of those 512.
repro_texts = eval_texts[:512]
repro_ds = build_text_dataset(repro_texts, llama_base_tok, MAX_LENGTH)
repro = compute_model_bpb(llama_base, repro_ds, count_utf8_bytes(repro_texts), torch)
print(f"Reproduced base Llama BPB (512 rows, trunc max_length=256): {repro.bits_per_byte:.4f}")
print("Report  base Llama BPB (2A4, same capped config)          : 0.7685")

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

ValueError: Experiment tokenizer must define a PAD token for model integration.

### Sanity check: full-coverage equals the library metric when nothing truncates

On texts that fit within the token budget for both tokenizers, no truncation happens, so
`compute_model_bpb_full` must equal `compute_model_bpb`. We assert this on a short-text subset
(base Llama), validating the new helper against the existing one.

In [ ]:
budget = MAX_LENGTH - 1
short_texts = [
    t for t in eval_texts[:400]
    if len(akan_tok(t, add_special_tokens=False)["input_ids"]) <= budget
    and len(llama_base_tok(t, add_special_tokens=False)["input_ids"]) <= budget
][:64]
print(f"Short non-truncated texts used for the check: {len(short_texts)}")

old_ds = build_text_dataset(short_texts, llama_base_tok, MAX_LENGTH)
old_bpb = compute_model_bpb(llama_base, old_ds, count_utf8_bytes(short_texts), torch).bits_per_byte
full_bpb = compute_model_bpb_full(llama_base, short_texts, llama_base_tok, torch, CHUNK_SIZE)["bits_per_byte"]
print(f"old (library)     : {old_bpb:.6f}")
print(f"full (this helper): {full_bpb:.6f}")
assert abs(old_bpb - full_bpb) < 1e-3, "full-coverage helper diverged from the library metric on non-truncated texts"
print("OK - helper matches the library metric when nothing truncates.")

## E. Corrected BASE BPB (full byte coverage, no adapters)

For each base model: the **original** truncating BPB on the full 2,500 rows vs the
**corrected** full-coverage BPB. The base-outlier question needs no adapters — just the base
model off the Hub. Headline check: does Llama's base BPB rise substantially from 0.7685?

In [ ]:
def score_base(name, model_id, model=None, tokenizer=None):
    own_model = model is None
    if tokenizer is None:
        tokenizer = base_toks.get(name) or AutoTokenizer.from_pretrained(model_id)
    if model is None:
        model = _load_base_model_for_bpb(model_id, AutoModelForCausalLM, torch, quantized=True)
    _set_model_token_config(model, tokenizer)  # align pad/eos/bos config (mirrors compute_bpb_metrics)
    old_full = compute_model_bpb(
        model, build_text_dataset(eval_texts, tokenizer, MAX_LENGTH), EVAL_BYTES, torch
    ).bits_per_byte
    corrected = compute_model_bpb_full(model, eval_texts, tokenizer, torch, CHUNK_SIZE)["bits_per_byte"]
    if own_model:
        del model
        free()
    return {"old_trunc_full2500": round(old_full, 4), "corrected_full": round(corrected, 4),
            "delta": round(corrected - old_full, 4)}


base_bpb = {}
# Reuse the already-loaded Llama base (from cell D) to save a reload.
base_bpb["Llama-3.2-1B"] = score_base("Llama-3.2-1B", LLAMA_ID, model=llama_base, tokenizer=llama_base_tok)
del llama_base
free()

for name, mid in BASE_MODELS.items():
    if name == "Llama-3.2-1B":
        continue
    if name not in base_toks:
        print(f"  [skip] {name}: base tokenizer unavailable")
        continue
    try:
        base_bpb[name] = score_base(name, mid)
    except Exception as exc:  # noqa: BLE001
        print(f"  [skip] {name}: {exc}")

base_df = pd.DataFrame(base_bpb).T
print("Base BPB: original (truncating) vs corrected (full coverage), full 2,500 rows")
base_df

## F. Retrain the Llama Akan adapter in-session (both arms)

Reproduce the 2A4 adapters from code (seed 42) — no Kaggle retrieval. Same hyperparameters as
the 2A4 run; only `--embedding-init-mode` differs. Adapters are written to `*_helper` dirs so
they never collide with the committed run. (`meta-llama/Llama-3.2-1B` is already on the
`colab-qlora` allowlist.)

In [ ]:
# Arm A - random embedding init
!python scripts/model_integration.py \
    --experiment-id phase2a4_llama_3_2_1b_tts_helper \
    --model-id meta-llama/Llama-3.2-1B \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a4_llama_3_2_1b_tts_helper \
    --results-output results/phase2a4_llama_3_2_1b_tts_helper.json \
    --device-mode colab-qlora \
    --embedding-init-mode random \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

In [ ]:
# Arm B - mean-of-subword embedding init
!python scripts/model_integration.py \
    --experiment-id phase2a4_llama_3_2_1b_tts_meansub_helper \
    --model-id meta-llama/Llama-3.2-1B \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a4_llama_3_2_1b_tts_meansub_helper \
    --results-output results/phase2a4_llama_3_2_1b_tts_meansub_helper.json \
    --device-mode colab-qlora \
    --embedding-init-mode mean_subword \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

## G. Corrected EXPERIMENT BPB + corrected improvement

Reload each saved adapter following the same sequence as
`verify_saved_qwen_artifacts` (resize embeddings, then `PeftModel.from_pretrained`, with the
resized `embed_tokens`/`lm_head` restored via `modules_to_save`). Score the fine-tuned model
with full byte coverage using **its own Akan tokenizer** (reloaded from the adapter dir), then
compute the corrected `base - experiment` against the corrected Llama base BPB from cell E.

In [ ]:
def reload_adapter(output_dir, model_id):
    """Mirror akan_bpe.model_integration.verify_saved_qwen_artifacts reload sequence."""
    tok = AutoTokenizer.from_pretrained(output_dir)
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token
    qc = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=qc, dtype="auto")
    base.config.tie_word_embeddings = False
    _set_model_token_config(base, tok)
    base.resize_token_embeddings(len(tok), pad_to_multiple_of=64)
    model = PeftModel.from_pretrained(base, output_dir)
    _set_model_token_config(model, tok)
    model.eval()
    return model, tok


ARMS = {
    "random": "models/phase2a4_llama_3_2_1b_tts_helper",
    "mean_subword": "models/phase2a4_llama_3_2_1b_tts_meansub_helper",
}
base_corrected = base_bpb["Llama-3.2-1B"]["corrected_full"]
base_trunc_2500 = base_bpb["Llama-3.2-1B"]["old_trunc_full2500"]

exp_bpb = {}
for arm, out_dir in ARMS.items():
    model, tok = reload_adapter(out_dir, LLAMA_ID)
    corrected = compute_model_bpb_full(model, eval_texts, tok, torch, CHUNK_SIZE)["bits_per_byte"]
    # Row-count-matched truncating BPB: same metric as the original run but on all 2,500 rows.
    # Holding row count fixed isolates coverage correction from sample-size change.
    trunc_2500 = compute_model_bpb(
        model, build_text_dataset(eval_texts, tok, MAX_LENGTH), EVAL_BYTES, torch
    ).bits_per_byte
    exp_bpb[arm] = {
        "experiment_corrected": round(corrected, 4),
        "experiment_trunc_2500": round(trunc_2500, 4),
        "base_corrected": round(base_corrected, 4),
        "base_trunc_2500": round(base_trunc_2500, 4),
        "improvement_corrected": round(base_corrected - corrected, 4),
        "improvement_trunc_2500": round(base_trunc_2500 - trunc_2500, 4),
    }
    del model
    free()

exp_df = pd.DataFrame(exp_bpb).T
print("Corrected experiment BPB (full coverage) and corrected improvement vs base")
exp_df

## H. Summary — original vs truncation-corrected

Side-by-side of the reported (512-row, truncating) 2A4 numbers and the corrected
(full 2,500-row, full-coverage) numbers, with the verdict on whether Llama flips positive.

In [ ]:
# Reported 2A4 numbers (report section 11; 512-row capped, truncating BPB).
REPORTED = {
    "base":         0.7685,
    "random":       {"experiment": 1.0357, "improvement": -0.2672},
    "mean_subword": {"experiment": 0.8966, "improvement": -0.1281},
}

summary = []
for arm in ("random", "mean_subword"):
    summary.append({
        "arm": arm,
        # reported: 512 rows, truncating metric
        "base_reported":    REPORTED["base"],
        "exp_reported":     REPORTED[arm]["experiment"],
        "improv_reported":  REPORTED[arm]["improvement"],
        # trunc/2500: truncating metric, 2500 rows — isolates row-count effect from coverage fix
        "base_trunc_2500":  exp_bpb[arm]["base_trunc_2500"],
        "exp_trunc_2500":   exp_bpb[arm]["experiment_trunc_2500"],
        "improv_trunc_2500": exp_bpb[arm]["improvement_trunc_2500"],
        # full/2500: full coverage, 2500 rows — the corrected result
        "base_corrected":   exp_bpb[arm]["base_corrected"],
        "exp_corrected":    exp_bpb[arm]["experiment_corrected"],
        "improv_corrected": exp_bpb[arm]["improvement_corrected"],
    })
summary_df = pd.DataFrame(summary).set_index("arm")
print("2A4 Llama: reported (trunc/512) → trunc/2500 → full-coverage/2500")
print(summary_df.to_string())

flipped_trunc = exp_bpb["mean_subword"]["improvement_trunc_2500"] > 0
flipped = exp_bpb["mean_subword"]["improvement_corrected"] > 0
print()
print(f"mean_subword trunc/2500   : {exp_bpb['mean_subword']['improvement_trunc_2500']:+.4f}"
      f"  ->  {'flips positive (row-count alone sufficient)' if flipped_trunc else 'still negative — row count not the cause'}")
print(f"mean_subword full-coverage: {exp_bpb['mean_subword']['improvement_corrected']:+.4f}"
      f"  ->  {'FLIPS POSITIVE (Akan wins after correction)' if flipped else 'still negative'}")
print()
print("If improv_trunc_2500 stays negative but improv_corrected flips positive, the flip is")
print("attributable to coverage correction alone — not to the change in eval-set size.")
print()
print("Note: every base tokenizer has higher fertility than the Akan tokenizer, so the same")
print("truncation correction should raise the Akan advantage at 2A1-2A3 too. If confirmed, the")
print("real fix is in the library (build_text_dataset / compute_model_bpb -> full coverage) plus")
print("a ladder re-run and a report.md update.")

### Optional cross-checks

Two independent robustness checks for the corrected BPB result:

1. **Chunk size 256 vs 512** (`compute_model_bpb_full`): halving boundary density tests whether
   the chunk cold-start changes the BPB quantitatively.
2. **Sliding window** (`compute_model_bpb_sliding`, window=512, stride=256): eliminates cold-start
   bias entirely by giving every new target ≥255 tokens of left context. If the sliding-window
   improvement agrees in *sign* with the non-overlapping result, boundary bias is not driving
   the flip.

In [ ]:
# --- Cross-check 1: chunk_size 256 vs 512 ---
llama_base2 = _load_base_model_for_bpb(LLAMA_ID, AutoModelForCausalLM, torch, quantized=True)
_set_model_token_config(llama_base2, llama_base_tok)
for cs in (256, 512):
    r = compute_model_bpb_full(llama_base2, eval_texts, llama_base_tok, torch, chunk_size=cs)
    print(f"corrected base Llama BPB @ chunk_size={cs}: {r['bits_per_byte']:.4f}")
del llama_base2
free()

# --- Cross-check 2: sliding window (window=512, stride=256) ---
# Every new target has >=255 tokens of left context within its window — no cold-start.
# Reloads Llama base + mean_subword adapter once more (~same cost as cross-check 1).
print()
print("Sliding-window confirmation (window=512, stride=256) ...")
llama_base3 = _load_base_model_for_bpb(LLAMA_ID, AutoModelForCausalLM, torch, quantized=True)
_set_model_token_config(llama_base3, llama_base_tok)
sliding_base = compute_model_bpb_sliding(llama_base3, eval_texts, llama_base_tok, torch)
del llama_base3; free()
print(f"  Base Llama (sliding w=512 s=256)        : {sliding_base['bits_per_byte']:.4f}")
print(f"  Base Llama (non-overlapping chunk=256)  : {base_bpb['Llama-3.2-1B']['corrected_full']:.4f}")

model_sw, tok_sw = reload_adapter(ARMS["mean_subword"], LLAMA_ID)
sliding_exp = compute_model_bpb_sliding(model_sw, eval_texts, tok_sw, torch)
del model_sw; free()
print(f"  Exp mean_subword (sliding)              : {sliding_exp['bits_per_byte']:.4f}")
print(f"  Exp mean_subword (non-overlapping)      : {exp_bpb['mean_subword']['experiment_corrected']:.4f}")
sliding_improv = sliding_base["bits_per_byte"] - sliding_exp["bits_per_byte"]
print(f"  Improvement (sliding)                   : {sliding_improv:+.4f}")
print(f"  Improvement (non-overlapping)           : {exp_bpb['mean_subword']['improvement_corrected']:+.4f}")
qualitative_agree = (sliding_improv > 0) == (exp_bpb["mean_subword"]["improvement_corrected"] > 0)
print(f"\n  Qualitative agreement: "
      f"{'YES — boundary cold-start is not driving the flip' if qualitative_agree else 'NO — results diverge; chunk-boundary bias may be significant'}")